# SQL4Business Deliverable 2

This notebook is self-contained for Google Colab. It uploads the current repository ZIP, loads Qwen2.5-Coder-3B-Instruct once, runs the direct baseline and the structured solution on the same 15 questions, and writes the comparison trace.

Before the first code cell, select `Runtime > Change runtime type > T4 GPU`. Upload the repository ZIP when requested. Do not run cells from an older notebook version.

In [ ]:
from google.colab import files
from pathlib import Path
import io
import os
import shutil
import sys
import zipfile

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
extract_root = Path('/content/sql4business_runtime')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
    archive.extractall(extract_root)

matches = [path for path in extract_root.rglob('business.db') if path.parent.name == 'data']
if not matches:
    raise RuntimeError('The uploaded ZIP does not contain data/business.db.')

REPO_ROOT = matches[0].parent.parent
assert (REPO_ROOT / 'data' / 'questions.json').exists()
assert (REPO_ROOT / 'src').exists()
assert (REPO_ROOT / 'scripts').exists()

os.chdir(REPO_ROOT)
for module_name in list(sys.modules):
    if module_name == 'sql4business' or module_name.startswith('sql4business.'):
        del sys.modules[module_name]
sys.path.insert(0, str(REPO_ROOT / 'src'))

print('Repository:', REPO_ROOT)
print('ZIP:', zip_name)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
pipeline_text = (REPO_ROOT / 'src' / 'sql4business' / 'pipeline.py').read_text(encoding='utf-8')
sql_tools_text = (REPO_ROOT / 'src' / 'sql4business' / 'sql_tools.py').read_text(encoding='utf-8')
print('updated retry diagnostics:', 'Result sets:' in pipeline_text)
print('date validation:', '_validate_date_literals' in sql_tools_text)
print('direct-plan normalization:', 'Multiple direct steps reduced to the final step.' in pipeline_text)
assert 'Result sets:' in pipeline_text
assert '_validate_date_literals' in sql_tools_text
assert 'Multiple direct steps reduced to the final step.' in pipeline_text

In [ ]:
import json
from sql4business.model import GenerationSettings, HuggingFaceGenerator
from sql4business.pipeline import BusinessAssistant

DB_PATH = REPO_ROOT / 'data' / 'business.db'
QUESTIONS_PATH = REPO_ROOT / 'data' / 'questions.json'
dataset = json.loads(QUESTIONS_PATH.read_text(encoding='utf-8'))
questions = dataset['questions']
MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
generator = HuggingFaceGenerator(MODEL_ID, GenerationSettings(max_new_tokens=384))
assistant = BusinessAssistant(DB_PATH, generator, max_retries=2)
print(f'{len(questions)} questions loaded; model={MODEL_ID}')

In [ ]:
# End-to-end inspection on q09, a combined question.
question = next(item for item in questions if item['id'] == 'q09')
try:
    result = assistant.answer(question['question'])
    print('QUESTION:', question['question'])
    print('PLAN:', json.dumps(result.plan, ensure_ascii=False, indent=2))
    for index, execution in enumerate(result.executions, start=1):
        print(f'STEP {index}:', execution.sql)
        print('ROWS:', execution.rows)
    print('COMPOSED:', result.composed.value)
    print('REPORT:', result.report)
    print('REPORT FIDELITY:', result.report_fidelity)
except Exception as error:
    print('q09 failed:', error)

In [ ]:
# Full comparison using the already loaded model.
# Set EVAL_LIMIT = 1 for a smoke test, or 0 for all 15 questions.
EVAL_LIMIT = 0

import time
from sql4business.evaluation import answer_matches, execute_baseline_output, gold_composed_answer, matches_gold_results
from sql4business.model import direct_prompt
from sql4business.sql_tools import connect_read_only

eval_questions = questions[:EVAL_LIMIT] if EVAL_LIMIT else questions
comparison_path = REPO_ROOT / 'results' / 'deliverable2_comparison.json'
records = []

for index, question in enumerate(eval_questions, start=1):
    print(f'[{index}/{len(eval_questions)}] {question["id"]}: baseline...', flush=True)
    baseline_started = time.perf_counter()
    baseline_raw = generator.generate(direct_prompt(dataset['schema'], question['question']))
    conn = connect_read_only(DB_PATH)
    try:
        baseline_executions = execute_baseline_output(conn, baseline_raw)
    finally:
        conn.close()
    baseline_elapsed = time.perf_counter() - baseline_started
    baseline_correct = matches_gold_results([item.rows for item in baseline_executions], question['gold_result'])

    print(f'[{index}/{len(eval_questions)}] {question["id"]}: structured solution...', flush=True)
    solution_started = time.perf_counter()
    try:
        solution = assistant.answer(question['question'])
        solution_elapsed = time.perf_counter() - solution_started
        solution_sql_correct = matches_gold_results([item.rows for item in solution.executions], question['gold_result'])
        solution_answer_correct = answer_matches(solution.composed.value, gold_composed_answer(question, DB_PATH))
        solution_full_correct = solution_sql_correct and solution_answer_correct and solution.report_fidelity
        records.append({
            'id': question['id'], 'type': question['type'], 'question': question['question'],
            'baseline': {'model_output': baseline_raw, 'execution_correct': baseline_correct, 'duration_seconds': baseline_elapsed},
            'solution': {
                'plan': solution.plan,
                'executions': [{'sql': item.sql, 'rows': item.rows, 'error': item.error} for item in solution.executions],
                'composed_answer': solution.composed.value, 'report': solution.report,
                'execution_correct': solution_sql_correct, 'answer_correct': solution_answer_correct,
                'report_fidelity': solution.report_fidelity, 'full_correct': solution_full_correct,
                'duration_seconds': solution_elapsed, 'attempts': solution.attempts, 'errors': solution.errors,
            },
        })
        print(f'[{index}/{len(eval_questions)}] {question["id"]}: done, full_correct={solution_full_correct}', flush=True)
    except Exception as error:
        records.append({
            'id': question['id'], 'type': question['type'], 'question': question['question'],
            'baseline': {'model_output': baseline_raw, 'execution_correct': baseline_correct, 'duration_seconds': baseline_elapsed},
            'solution': {'full_correct': False, 'error': str(error)},
        })
        print(f'[{index}/{len(eval_questions)}] {question["id"]}: solution failed: {error}', flush=True)

comparison_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', comparison_path)

for group in ('puntual', 'combinada'):
    subset = [record for record in records if record['type'] == group]
    if not subset:
        print(f'{group}: no questions in this run')
        continue
    baseline_score = sum(record['baseline'].get('execution_correct', False) for record in subset) / len(subset)
    solution_score = sum(record['solution'].get('execution_correct', False) for record in subset) / len(subset)
    full_score = sum(record['solution'].get('full_correct', False) for record in subset) / len(subset)
    print(f'{group}: baseline SQL={baseline_score:.2%}, solution SQL={solution_score:.2%}, solution full={full_score:.2%}')

In [ ]:
# Fill the measured values and first real failure into the one-page LaTeX document.
import subprocess
subprocess.run([
    sys.executable, str(REPO_ROOT / 'scripts' / 'update_deliverable2_tex.py'),
    '--results', str(comparison_path),
    '--tex', str(REPO_ROOT / 'docs' / 'deliverable2.tex'),
], check=True)
print(REPO_ROOT / 'docs' / 'deliverable2.tex')

The comparison uses the same SQL-result criterion for baseline and solution. The structured solution additionally checks its composed answer and report fidelity. The output JSON and updated LaTeX file are traceable to this run.